In [1]:
#安装依赖（Kaggle）
import subprocess
import sys

def install_dependencies():
    """安装所需依赖包"""
    packages = [
        "transformers>=4.37.0",
        "torch>=2.0.0",
        "rich",
        "accelerate",
        "sentencepiece",
        "protobuf",
        "bitsandbytes>=0.41.0",  
        "peft"
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

install_dependencies()

# 导入核心库
import re
import json
import torch
from rich import print
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    BitsAndBytesConfig  
)


schema = {
    '科技': ['产品名称', '发布日期', '技术类型', '版本号', '适用场景'],
}

# 信息抽取的模版
IE_PATTERN = "{}\n\n提取上述句子中{}的实体，并按照JSON格式输出，上述句子中不存在的信息用['原文中未提及']来表示，多个值之间用','分隔。"

# 科技行业示例数据
ie_examples = {
    '科技': [
        {
            'content': '2024-03-15，阿里云正式发布新一代云原生数据库PolarDB 8.0版本，该产品基于分布式架构技术，主要适用于电商平台的高并发数据处理场景。',
            'answers': {
                '产品名称': ['PolarDB'],
                '发布日期': ['2024-03-15'],
                '技术类型': ['分布式架构'],
                '版本号': ['8.0'],
                '适用场景': ['电商平台高并发数据处理'],
            }
        }
    ]
}

# 定义init_prompts函数
def init_prompts(tokenizer):
    """
    初始化前置prompt，构建Qwen格式的对话历史，便于模型做incontext learning。
    """
    # Qwen的对话格式是[{"role": "user", "content": ...}, {"role": "assistant", ...}]
    ie_pre_history = []
    
    # 初始指令
    system_prompt = {
        "role": "user",
        "content": "现在你需要帮助我完成信息抽取任务，当我给你一个句子时，你需要帮我抽取出句子中科技行业的实体信息，并按照JSON的格式输出，上述句子中没有的信息用['原文中未提及']来表示，多个值之间用','分隔。"
    }
    system_response = {
        "role": "assistant",
        "content": "好的，请输入您的句子。"
    }
    ie_pre_history.append(system_prompt)
    ie_pre_history.append(system_response)
    
    # 添加示例
    for _type, example_list in ie_examples.items():
        for example in example_list:
            sentence = example["content"]
            properties_str = ', '.join(schema[_type])
            schema_str_list = f'"{_type}"({properties_str})'
            sentence_with_prompt = IE_PATTERN.format(sentence, schema_str_list)
            
            # 添加示例的用户提问和助手回答
            ie_pre_history.append({"role": "user", "content": sentence_with_prompt})
            ie_pre_history.append({
                "role": "assistant",
                "content": json.dumps(example['answers'], ensure_ascii=False)
            })
    
    # 应用Qwen的chat template，生成模型输入的文本
    prompt_text = tokenizer.apply_chat_template(
        ie_pre_history,
        tokenize=False,
        add_generation_prompt=False  # 暂时不加生成提示，后续拼接新问题时加
    )
    return {"ie_pre_prompt": prompt_text}

def clean_response(response: str):
    """
    后处理模型输出，提取JSON并格式化。
    """
    # 清理模型输出中的多余内容
    if '```json' in response:
        res = re.findall(r'```json(.*?)```', response, re.DOTALL)
        if len(res) and res[0]:
            response = res[0].strip()
    # 替换中文顿号为英文逗号
    response = response.replace('、', ',')
    # 尝试解析JSON
    try:
        return json.loads(response)
    except:
        # 解析失败时返回原始内容并提示
        print(f"[bold yellow]JSON解析失败，原始响应：{response}[/bold yellow]")
        return response

def inference(sentences: list, custom_settings: dict, tokenizer, model):
    """
    推理函数（适配Qwen模型）。
    
    Args:
        sentences (List[str]): 待抽取的句子。
        custom_settings (dict): 初始设定，包含few-shot示例的prompt。
        tokenizer: 分词器
        model: 加载好的Qwen模型
    """
    # 设备配置（Kaggle自动识别GPU/CPU）
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"使用设备：{device}")
    
    for sentence in sentences:
        cls_res = "科技"  
        if cls_res not in schema:
            print(f'The type model inferenced {cls_res} which is not in schema dict, exited.')
            return
        
        # 构建当前问题的prompt
        properties_str = ', '.join(schema[cls_res])
        schema_str_list = f'"{cls_res}"({properties_str})'
        current_question = IE_PATTERN.format(sentence, schema_str_list)
        
        # 拼接历史prompt和当前问题
        full_prompt = custom_settings["ie_pre_prompt"] + tokenizer.apply_chat_template(
            [{"role": "user", "content": current_question}],
            tokenize=False,
            add_generation_prompt=True  # 加上生成提示，让模型开始回答
        )
        
        # 分词并转换为模型输入
        inputs = tokenizer([full_prompt], return_tensors="pt").to(device)
        
        # 模型生成（适配Qwen的生成参数）
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,  # 增大生成长度，确保能输出完整JSON
                temperature=0.1,     # 低温度保证输出稳定
                do_sample=False,     # 贪心解码
                top_p=0.9,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
            )
        
        # 提取生成的回复（去掉输入部分）
        response = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
        ie_res = clean_response(response)
        
        # 打印结果
        print(f'>>> [bold bright_red]sentence: {sentence}')
        print(f'>>> [bold bright_green]inference answer:{ie_res} \n')

if __name__ == '__main__':
    # 模型名称（Kaggle中直接使用Hugging Face Hub的模型）
    model_name_or_path = "Qwen/Qwen2.5-7B-Instruct"
    
    # 配置4bit量化（核心修复：通过BitsAndBytesConfig传递量化参数）
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_path,
        trust_remote_code=True,
        use_fast=False
    )
    # 设置pad_token（避免生成时报错）
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 加载模型（修复：通过quantization_config传递4bit参数）
    model = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        quantization_config=bnb_config,  # 核心修复：用这个参数传递量化配置
        low_cpu_mem_usage=True
    )
    model.eval()  # 评估模式
    
    # 科技行业测试句子（核心修改3：自拟科技行业测试文本）
    sentences = [
        '2024-05-20，字节跳动发布AI开发工具Coze 2.5版本，该工具基于大语言模型技术，主要适用于企业级智能客服的快速搭建场景。',
        '2024-01-10，华为推出昇腾910B AI芯片，该产品采用7nm制程技术，版本号为V3.0，主要适用于超大规模算力中心的AI训练场景。',
    ]
    
    # 初始化prompt
    custom_settings = init_prompts(tokenizer)
    
    # 执行推理
    inference(sentences, custom_settings, tokenizer, model)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.8 MB/s eta 0:00:00


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


>>> sentence: 2024-05-20，字节跳动发布AI开发工具Coze 
2.5版本，该工具基于大语言模型技术，主要适用于企业级智能客服的快速搭建场景。

>>> inference answer:{'产品名称': ['Coze'], '发布日期': ['2024-05-20'], '技术类型': ['大语言模型'], '版本号': 
['2.5'], '适用场景': ['企业级智能客服快速搭建']} 

>>> sentence: 2024-01-10，华为推出昇腾910B 
AI芯片，该产品采用7nm制程技术，版本号为V3.0，主要适用于超大规模算力中心的AI训练场景。

>>> inference answer:{'产品名称': ['昇腾910B AI芯片'], '发布日期': ['2024-01-10'], '技术类型': ['7nm制程技术'], 
'版本号': ['V3.0'], '适用场景': ['超大规模算力中心AI训练']} 